# Repro Paper 2: *Edge-AI Smishing Detection with Weighted BCE*
**Venue:** Journal of Information and Telecommunication, 2026 (M041)
**Author:** Huynh Cong Phap (DOI: 10.1080/24751839.2026.2713250)
**Dataset:** UCI SMS Spam Collection ($N=5,574$), 80/20 train/test split ($N_{\text{test}}=1,115$: 965 ham, 150 spam).

### Primary Literature Quotes & Implementation Details:
1. **Model & Pooling:** *"combining a Bidirectional Gated Recurrent Unit (BiGRU) encoder with Masked Global Pooling to strictly exclude padding noise from pooled representations"* (H. C. Phap, JIT 2026).
2. **Hardware Profiling:** *"requires only a 5.21 MB serialized footprint and 0.25 ms CPU latency, compared to 256.02 MB and 40.78 ms for DistilBERT"* (H. C. Phap, JIT 2026).
3. **Loss Function (WBCE):**
   - The original paper formulates inverse class weighting as $w_1 = N / (2N_1)$ and $w_0 = N / (2N_0)$.
   - In PyTorch `nn.BCEWithLogitsLoss(pos_weight=...)`, the relative positive weight equals $w_1 / w_0 = N_0 / N_1 = 4,827 / 747 \approx 6.46$ (derived for the PyTorch implementation; this is an implementation derivation, not a quotation).
4. **Evaluation Setup:** The original paper evaluates across 5 stratified folds/seeds. Here we use a single pre-registered 80/20 stratified split ($N_{\text{test}}=1,115$: 965 ham, 150 spam) and profile CPU latency.

> **Known deviation:** a single split gives one realization of the test set, so the WBCE-vs-BCE comparison is a paired comparison on the same $N=1,115$ rows, not a cross-fold mean. Do not report it as a 5-fold replication.

In [ ]:
# Cell 1: Environment Setup & Dataset Ingestion (with mirror fallback)
!pip install -q torch scikit-learn pandas numpy xgboost transformers

import io
import json
import os
import pickle
import time
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
MIRROR_TSV_URL = (
    "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
)

def load_uci_sms() -> pd.DataFrame:
    # Primary source: the UCI archive itself.
    try:
        zip_path = "sms_spam.zip"
        if not os.path.exists(zip_path):
            urllib.request.urlretrieve(UCI_ZIP_URL, zip_path)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall("uci_sms")
        frame = pd.read_csv("uci_sms/SMSSpamCollection", sep="\t", names=["label_str", "text"])
        print("Source: UCI archive (SMSSpamCollection)")
        return frame
    except Exception as exc:  # noqa: BLE001
        print(f"[warn] UCI archive unavailable ({type(exc).__name__}: {exc}); trying mirror")

    # Fallback: public TSV mirror with a header row.
    try:
        with urllib.request.urlopen(MIRROR_TSV_URL, timeout=60) as resp:
            mirror = pd.read_csv(io.StringIO(resp.read().decode("utf-8")), sep="\t")
        mirror = mirror.rename(columns={"message": "text", "label": "label_str"})
        print("Source: GitHub TSV mirror (justmarkham/pycon-2016-tutorial)")
        return mirror[["label_str", "text"]]
    except Exception as exc:  # noqa: BLE001
        raise RuntimeError(
            "Could not download the UCI SMS Spam Collection from either source. "
            "Attach the Kaggle 'sms-spam-collection-dataset' as an input and set "
            "UCI_SMS_PATH to the local CSV."
        ) from exc

local_path = os.environ.get("UCI_SMS_PATH")
if local_path and os.path.exists(local_path):
    df = pd.read_csv(local_path, sep="\t", names=["label_str", "text"])
    print(f"Source: local file {local_path}")
else:
    df = load_uci_sms()

df = df.dropna(subset=["text"]).reset_index(drop=True)
df["label"] = (df["label_str"].str.strip().str.lower() == "spam").astype(int)
assert len(df) == 5574, f"Dataset size mismatch: {len(df)} != 5574"
print(f"UCI SMS verified: N={len(df)} | ham={(df['label'] == 0).sum()} | "
      f"spam={(df['label'] == 1).sum()}")

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
assert len(test_df) == 1115, f"Expected 1,115 test rows, got {len(test_df)}"
observed_counts = test_df["label"].value_counts().sort_index().to_dict()
if observed_counts != {0: 965, 1: 150}:
    print(f"[warn] test class counts {observed_counts} differ from the protocol's "
          f"{{0: 965, 1: 150}}; the integer counts below remain authoritative.")
print(f"Split verified: Train={len(train_df)} | Test={len(test_df)} {observed_counts}")

In [ ]:
# Cell 2: Tokenization & CostSensitiveBiGRU Architecture
# Vocabulary construction is fitted on the TRAIN split only (leakage prevention).
word_counts = {}
for text in train_df["text"]:
    for token in str(text).lower().split():
        word_counts[token] = word_counts.get(token, 0) + 1

VOCAB_SIZE = 15000
sorted_vocab = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:VOCAB_SIZE - 2]
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in sorted_vocab:
    vocab[word] = len(vocab)

def text_pipeline(text, max_len=100):
    tokens = [vocab.get(w, 1) for w in str(text).lower().split()][:max_len]
    length = max(len(tokens), 1)
    padded = tokens + [0] * (max_len - len(tokens))
    return padded, length

class SMSDataset(Dataset):
    def __init__(self, frame):
        self.samples = [text_pipeline(t) for t in frame["text"]]
        self.labels = frame["label"].values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        padded, length = self.samples[idx]
        return (torch.tensor(padded, dtype=torch.long),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.float))

train_loader = DataLoader(SMSDataset(train_df), batch_size=64, shuffle=True)
test_loader = DataLoader(SMSDataset(test_df), batch_size=64, shuffle=False)

class CostSensitiveBiGRU(nn.Module):
    # BiGRU + Masked Global Max & Average Pooling (H. C. Phap, 2026).
    # Dual-branch pooling is concatenated: in_features = hidden_dim * 4.
    def __init__(self, vocab_sz=15000, embed_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_sz, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 4, 1)

    def forward(self, input_ids, lengths):
        emb = self.embedding(input_ids)
        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        # total_length keeps the time dimension consistent when batch_size == 1.
        out, _ = pad_packed_sequence(
            packed_out, batch_first=True, total_length=input_ids.size(1)
        )

        mask = (input_ids != 0).unsqueeze(-1)
        out_masked_for_max = out.masked_fill(~mask, -1e9)
        max_pool = torch.max(out_masked_for_max, dim=1)[0]

        out_masked_for_avg = out.masked_fill(~mask, 0.0)
        lens = lengths.unsqueeze(1).float().clamp(min=1.0)
        avg_pool = torch.sum(out_masked_for_avg, dim=1) / lens

        pooled = torch.cat([max_pool, avg_pool], dim=1)
        return self.fc(pooled).squeeze(-1)

temp_model = CostSensitiveBiGRU()
total_params = sum(p.numel() for p in temp_model.parameters())
raw_weight_mb = total_params * 4 / (1024 * 1024)
print(f"CostSensitiveBiGRU Parameter Count: {total_params:,} parameters")
print(f"Calculated Raw FP32 Weight Size: {raw_weight_mb:.2f} MiB")
print("Note: the paper reports 5.21 MB; the exact hidden/embedding dimensions are not "
      "fully disclosed, so the measured footprint here is reported as-is.")

In [ ]:
# Cell 3: Training & Loss Function Ablation (Standard BCE vs. WBCE)
def train_model(wbce_weight=1.0, epochs=8, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = CostSensitiveBiGRU().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    pos_weight = torch.tensor([wbce_weight], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for _ in range(epochs):
        model.train()
        for ids, lens, labels in train_loader:
            ids, lens, labels = ids.to(device), lens.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(ids, lens), labels)
            loss.backward()
            optimizer.step()
    return model

POS_WEIGHT = len(df[df["label"] == 0]) / len(df[df["label"] == 1])
print(f"Derived pos_weight = N_ham / N_spam = {POS_WEIGHT:.4f}")

print("Training Variant 1: BiGRU + Standard BCE (w=1.0)...")
model_bce = train_model(wbce_weight=1.0)

print(f"Training Variant 2: BiGRU + WBCE (w={POS_WEIGHT:.2f}, derived from the paper's weighting)...")
model_wbce = train_model(wbce_weight=POS_WEIGHT)

def evaluate_model(model):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for ids, lens, labels in test_loader:
            ids, lens = ids.to(device), lens.to(device)
            probs = torch.sigmoid(model(ids, lens))
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend((probs >= 0.5).int().cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels).astype(int), np.array(all_preds), np.array(all_probs)

y_true, preds_bce, probs_bce = evaluate_model(model_bce)
_, preds_wbce, probs_wbce = evaluate_model(model_wbce)

def full_metrics(y_t, y_p, name):
    tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0, 1]).ravel()
    return {
        "Variant": name,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "N": int(tn + fp + fn + tp),
        "Spam Recall (%)": round(recall_score(y_t, y_p) * 100, 2),
        "Precision (%)": round(precision_score(y_t, y_p, zero_division=0) * 100, 2),
        "F1 (%)": round(f1_score(y_t, y_p) * 100, 2),
        "Accuracy (%)": round(accuracy_score(y_t, y_p) * 100, 2),
        "FPR (%)": round(fp / max(fp + tn, 1) * 100, 2),
    }

rows = [
    full_metrics(y_true, preds_bce, "BiGRU + Standard BCE"),
    full_metrics(y_true, preds_wbce, "BiGRU + WBCE"),
]
df_ablation = pd.DataFrame(rows)
for row in rows:
    assert row["N"] == 1115, f"Denominator integrity violated: {row}"
display(df_ablation)
df_ablation.to_csv("Repro_Paper2_WBCE_Ablation.csv", index=False)

# Per-sample telemetry for the statistical audit.
pd.DataFrame({
    "idx": np.arange(len(y_true)),
    "gold": y_true,
    "pred_bce": preds_bce,
    "prob_bce": np.round(probs_bce, 6),
    "pred_wbce": preds_wbce,
    "prob_wbce": np.round(probs_wbce, 6),
}).to_csv("Repro_Paper2_per_sample_predictions.csv", index=False)

recall_gain = rows[1]["Spam Recall (%)"] - rows[0]["Spam Recall (%)"]
print(f"\nSpam recall change under WBCE: {recall_gain:+.2f} pp")
print("Paired decisions on identical rows -> McNemar is admissible here.")

In [ ]:
# Cell 4: Hardware Benchmarking (CPU Latency & Serialized Footprint)
torch.save(model_wbce.state_dict(), "bigru_wbce.pt")
measured_mb = os.path.getsize("bigru_wbce.pt") / (1024 * 1024)

cpu_model = model_wbce.to("cpu").eval()
test_samples = [text_pipeline(t) for t in test_df["text"].iloc[:500]]

with torch.no_grad():
    for _ in range(50):
        p0, l0 = test_samples[0]
        _ = cpu_model(torch.tensor([p0], dtype=torch.long), torch.tensor([l0], dtype=torch.long))

latencies_ns = []
with torch.no_grad():
    for p, l in test_samples:
        t0 = time.perf_counter_ns()
        _ = cpu_model(torch.tensor([p], dtype=torch.long), torch.tensor([l], dtype=torch.long))
        latencies_ns.append(time.perf_counter_ns() - t0)

mean_cpu_latency_ms = float(np.mean(latencies_ns)) / 1e6
p95_cpu_latency_ms = float(np.percentile(latencies_ns, 95)) / 1e6
latencies_cold_ns = []
with torch.no_grad():
    for p, l in test_samples[:100]:
        t0 = time.perf_counter_ns()
        _ = cpu_model(torch.tensor([p], dtype=torch.long), torch.tensor([l], dtype=torch.long))
        latencies_cold_ns.append(time.perf_counter_ns() - t0)

print("=== HARDWARE AUDIT: CostSensitiveBiGRU ===")
print(f"Serialized footprint (state_dict on disk): {measured_mb:.2f} MB")
print(f"CPU single-sample latency (warm, n=500): {mean_cpu_latency_ms:.3f} ms "
      f"(P95 {p95_cpu_latency_ms:.3f} ms)")
print(f"CPU single-sample latency (no warm-up, n=100): "
      f"{float(np.mean(latencies_cold_ns)) / 1e6:.3f} ms")
print("Paper reference: 5.21 MB and 0.25 ms (hardware unspecified in the paper).")
print("Report the measured numbers, not the paper's, and state the hardware used.")

In [ ]:
# Cell 5: Supplementary Baselines (DistilBERT & Classical ML) with MEASURED cost
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from datasets import Dataset

comparison_rows = [{
    "Model": "BiGRU + WBCE (H. C. Phap 2026)",
    "Spam Recall (%)": rows[1]["Spam Recall (%)"],
    "F1 (%)": rows[1]["F1 (%)"],
    "Footprint (MB)": round(measured_mb, 2),
    "CPU Latency (ms)": round(mean_cpu_latency_ms, 3),
    "Type": "Proposed Edge Architecture",
}]

# --- 1. DistilBERT: single training pass, single prediction pass ---
distil_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def dist_tok_fn(b):
    return distil_tok(b["text"], truncation=True, max_length=128)

tr_ds = Dataset.from_pandas(train_df[["text"]]).map(dist_tok_fn, batched=True)
te_ds = Dataset.from_pandas(test_df[["text"]]).map(dist_tok_fn, batched=True)

distil_mod = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
d_trainer = Trainer(
    model=distil_mod,
    args=TrainingArguments(
        output_dir="./distil_out",
        per_device_train_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=tr_ds,
)
d_trainer.train()

distil_logits = d_trainer.predict(te_ds).predictions
distil_preds = np.argmax(distil_logits, axis=-1)
distil_f1 = f1_score(test_df["label"].values, distil_preds) * 100
distil_recall = recall_score(test_df["label"].values, distil_preds) * 100

torch.save(distil_mod.state_dict(), "distilbert_uci.pt")
distil_mb = os.path.getsize("distilbert_uci.pt") / (1024 * 1024)

distil_cpu = distil_mod.to("cpu").eval()
sample_texts = test_df["text"].iloc[:100].tolist()
with torch.no_grad():
    for _ in range(10):
        _ = distil_cpu(**distil_tok(sample_texts[0], return_tensors="pt", truncation=True,
                                    max_length=128))
    d_lats = []
    for txt in sample_texts:
        inp = distil_tok(txt, return_tensors="pt", truncation=True, max_length=128)
        t0 = time.perf_counter_ns()
        _ = distil_cpu(**inp)
        d_lats.append(time.perf_counter_ns() - t0)
distil_cpu_ms = float(np.mean(d_lats)) / 1e6

comparison_rows.append({
    "Model": "DistilBERT (H. C. Phap 2026 baseline)",
    "Spam Recall (%)": round(float(distil_recall), 2),
    "F1 (%)": round(float(distil_f1), 2),
    "Footprint (MB)": round(distil_mb, 2),
    "CPU Latency (ms)": round(distil_cpu_ms, 3),
    "Type": "Official Transformer Baseline",
})

# --- 2. Classical ML: footprint and latency are MEASURED, not asserted ---
tfidf = TfidfVectorizer(max_features=5000)
X_tr = tfidf.fit_transform(train_df["text"])
X_te = tfidf.transform(test_df["text"])
y_tr = train_df["label"].values
y_te = test_df["label"].values

ml_models = {
    "TF-IDF + LR (Supplementary)": LogisticRegression(max_iter=1000),
    "TF-IDF + Linear SVM (Supplementary)": LinearSVC(),
    "TF-IDF + Random Forest (Supplementary)": RandomForestClassifier(
        n_estimators=100, random_state=42),
    "TF-IDF + XGBoost (Supplementary)": XGBClassifier(
        n_estimators=100, random_state=42, eval_metric="logloss"),
}

lat_probe = test_df["text"].iloc[:100].tolist()
for name, clf in ml_models.items():
    clf.fit(X_tr, y_tr)
    pred = clf.predict(X_te)
    blob = pickle.dumps({"vectorizer": tfidf, "clf": clf})
    for _ in range(5):
        _ = clf.predict(tfidf.transform([lat_probe[0]]))
    lats = []
    for txt in lat_probe:
        t0 = time.perf_counter_ns()
        _ = clf.predict(tfidf.transform([txt]))
        lats.append(time.perf_counter_ns() - t0)
    comparison_rows.append({
        "Model": name,
        "Spam Recall (%)": round(float(recall_score(y_te, pred)) * 100, 2),
        "F1 (%)": round(float(f1_score(y_te, pred)) * 100, 2),
        "Footprint (MB)": round(len(blob) / (1024 * 1024), 2),
        "CPU Latency (ms)": round(float(np.mean(lats)) / 1e6, 3),
        "Type": "Supplementary Baseline",
    })

res_df = pd.DataFrame(comparison_rows)
res_df.to_csv("Repro_Paper2_UCI_SMS_Comparison.csv", index=False)
display(res_df)
print("Footprint = pickled vectorizer+model (ML) or state_dict (neural); "
      "latency = single-message end-to-end predict on CPU, this machine.")